# 05 · Build the feature store (extract once, cache to parquet)

Feature extraction is ~2-4 s/log (88% is the changepoint QC). This notebook
extracts features for all logs **in parallel** and caches them to parquet so
you never recompute while iterating on the model.

- **Config-aware cache**: the store is keyed on a fingerprint of the extraction
  config. If you change anything that affects features (e.g. `cp_decimate`,
  frequency range, QC thresholds), the store is **wiped and rebuilt** on the
  next run. Downstream-only settings (geometry/labels, PLSR) do NOT invalidate
  it. Pass `force=True` to rebuild unconditionally.
- **Resumable**: within the *same* config, a `manifest.csv` records processed
  logs and re-running skips them — safe to interrupt a long run.
- **Speed knob**: `cfg.cp_decimate = 4` cuts extraction ~8x with no change to
  acceptance in tests. Use 1 for an exact, slowest run.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np, pandas as pd
from intelliswitch import Config
DATA_ROOT = r"/Volumes/Seagate Expansion Drive/Mohammad/M Drive DTU/MATLAB Scripts/IntelliswitchScripts"
W10  = os.path.join(DATA_ROOT, "W10v140")
BLADE = os.path.join(DATA_ROOT, "Bladenr8252")
STORE = "feature_store"      # output dir for the parquet cache

cfg = Config()
cfg.cp_decimate = 4          # 8x faster extraction; set 1 for exact
cfg.cp_decimate


## 1 · Build (or resume) the store
Set `LIMIT=None` for the full ~7,800 logs. Start with a few hundred to check.
`n_workers` defaults to all cores.


In [ ]:
from intelliswitch.pipeline import build_feature_store, load_feature_store, store_summary

LIMIT = 500          # None = all logs
build_feature_store(W10, STORE, cfg,
                    n_workers=os.cpu_count(),
                    shard_size=400,
                    limit=LIMIT)
store_summary(STORE)


## 2 · Load cached features + attach geometry labels
Labels are attached at load time (cheap) so geometry tweaks don't invalidate
the expensive feature cache.


In [ ]:
from intelliswitch.geometry import align_campaigns, build_deg_model
from intelliswitch.geometry.deg_model import attach_labels

feat = load_feature_store(STORE)
deg = build_deg_model(align_campaigns(BLADE, cfg), cfg)
data = attach_labels(feat, deg, cfg).dropna(subset=['y'])
print('feature rows:', len(feat), '| labelled rows:', len(data))
data.groupby('sensor').size()


## 3 · Train per-location PLSR on the cached data
(Same as notebook 04, but now instant to re-run since features are cached.)


In [ ]:
from intelliswitch.model import fit_per_location
results = fit_per_location(data, cfg)
rows = []
for name, r in results.items():
    if r['model'] is None:
        rows.append(dict(sensor=name, error=r.get('error'))); continue
    m, e = r['model'], r['eval']
    rows.append(dict(sensor=name, NOC=m.n_components, n_train=m.n_train,
                     train_R2=round(m.train_r2,3), test_pts=e['n_testpoints'],
                     MAPE=round(e['mape'],1), corr=round(e['corr'],3)))
pd.DataFrame(rows)


### Tip: verify the speed knob on your data
Compare `cp_decimate = 1` vs `4` on a handful of logs; if acceptance matches,
use 4 for the full run.
